In [ ]:
!pip install selenium==4.1.1

In [ ]:
!pip install transformers

In [ ]:
!pip install torch
!pip install torchvision
!pip install torchaudio



In [ ]:
!pip install jieba

In [ ]:
!pip install beautifulsoup4

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import csv
import time
import requests, json, re, csv
import jieba
from bs4 import BeautifulSoup
import torch
from torch.utils.data import DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
from sklearn.model_selection import train_test_split
import pandas as pd
import os

In [ ]:
#iver = webdriver.Chrome("chromedriver.exe")
#
#fault_url = "https://www.google.com.tw/maps/place/%E7%8B%82%E4%B8%80%E9%8D%8B%EF%BC%8D%E6%B7%A1%E6%B0%B4%E5%8C%97%E6%96%B0%E5%BA%97%EF%BC%8D%E9%B9%B9%E9%A6%99%E6%B9%AF%E9%A0%AD%E5%B0%88%E9%96%80%E5%BA%97/@25.1778886,121.4489728,17z/data=!3m2!4b1!5s0x3442affefc6fc1cb:0x6bb40dd005896167!4m6!3m5!1s0x3442afdeb962eba3:0xf91162320e6f815a!8m2!3d25.1778886!4d121.4489728!16s%2Fg%2F11r_gfkngf?hl=zh-TW&entry=ttu"
#iver.get(default_url)
#me.sleep(1)
#
#iver.find_element(By.XPATH, '//*[@id="searchboxinput"]').send_keys("狂一鍋－淡水北新店－鹹香湯頭專門店")
#me.sleep(1)
#
#iver.find_element(By.XPATH, '//*[@id="searchbox-searchbutton"]').click()
#me.sleep(5)
#
#iver.find_element(By.XPATH, '//*[@id="QA0Szd"]/div/div/div[1]/div[2]/div/div[1]/div/div/div[3]/div/div/button[2]').click()
#me.sleep(2)
#
#r i in range(0, 80):
#  pane = driver.find_element(By.CLASS_NAME, 'm6QErb.DxyBCb.kA9KIf.dS8AEf')
#  driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", pane)
#  time.sleep(1.5)
#
#l = driver.find_elements(By.CLASS_NAME,'jftiEf.fontBodyMedium')
#
#th open("content.csv", "w", encoding = "UTF-8", newline = "")as csvfile:
#  writer = csv.writer(csvfile)
#  for i in all:
#      try:
#          comment = i.find_element(By.CLASS_NAME,"wiI7pd").text.replace("\n","")
#
#          print(comment)
#          print("-------------------------------------------------------")
#
#          writer.writerow([comment])
#
#      except:
#          continue

In [ ]:
with open("positive.txt", "r", encoding = "utf-8") as f:
    positive_words = set(f.read().splitlines())

with open("negative.txt", "r", encoding = "utf-8") as f:
    negative_words = set(f.read().splitlines())

def score_sentence(sentence):
    words = jieba.lcut(sentence)

    positive_matches = [word for word in words if word in positive_words]
    negative_matches = [word for word in words if word in negative_words]

    score = len(positive_matches) - len(negative_matches)
    return score, words, positive_matches, negative_matches

output_rows = [["Sentence", "Score", "Sentiment"]]

with open("data.csv", "r", encoding = "utf-8") as file:
    reader = csv.reader(file)
    for row in reader:
        sentence = row[0]
        score, words, positive_matches, negative_matches = score_sentence(sentence)
        print(score, words, positive_matches, negative_matches)

        if score > 0:
            sentiment = "正評"

        elif score < 0:
            sentiment = "負評"

        else:
            sentiment = "中立"

        print(f"Sentence: {sentence}")
        #print(f"Segmented Words: {words}")
        #print(f"Positive Words: {positive_matches}")
        #print(f"Negative Words: {negative_matches}")
        print(f"Score: {score}, Sentiment: {sentiment}")
        print("\n")
        output_rows.append([sentence, score, sentiment])

    with open("score_comments.csv", "w", encoding = "utf-8", newline = '') as file:
        writer = csv.writer(file)
        writer.writerows(output_rows)

In [ ]:
# 設定參數
DATA_PATH = 'score_comments.csv'
MODEL_NAME = 'bert-base-chinese'
BATCH_SIZE = 8
EPOCHS = 16

In [ ]:
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [ ]:
# 載入和準備數據
def prepare_data(path):
    """從CSV檔案載入數據，刪除空值行，轉換標籤類型為整數，並分割資料集。"""
    data = pd.read_csv(path)
    data.dropna(subset=['Sentence', 'Sentiment'], inplace=True)  # 删除有空值的行
    data['Sentiment'] = data['Sentiment'].replace({'正評': 2, '負評': 1, '中立':0})
    train_texts, test_texts, train_labels, test_labels = train_test_split(
        data['Sentence'], data['Sentiment'], test_size=0.2, random_state=42)
    return train_texts, test_texts, train_labels, test_labels

# 自訂資料集類別
class TextDataset(torch.utils.data.Dataset):
    """準備文字資料供BERT模型使用。"""
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=max_len, return_tensors='pt')
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

# 訓練模型
def train_model(model, data_loader, optimizer, device):
    """在一個訓練週期內訓練BERT模型，計算損失與準確率。"""
    model.train()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    for batch in data_loader:
        batch = {k: v.to(device) for k, v in batch.items()}  # 將資料移至CPU或GPU
        outputs = model(**batch)  # 模型前向傳播
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()  # 反向傳播計算梯度
        optimizer.step()  # 更新模型參數
        optimizer.zero_grad()  # 清除舊梯度

        # 計算準確率
        preds = torch.argmax(outputs.logits, dim=-1)
        total_correct += (preds == batch['labels']).sum().item()
        total_samples += batch['labels'].size(0)

    # 計算平均損失和準確率
    avg_loss = total_loss / len(data_loader)
    accuracy = total_correct / total_samples
    return avg_loss, accuracy

# 主程式
def main():
    train_texts, test_texts, train_labels, test_labels = prepare_data(DATA_PATH)
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
    train_data = TextDataset(train_texts.tolist(), train_labels.tolist(), tokenizer)
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, num_workers=0)

    model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
    model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))  # 移動模型至GPU中
    optimizer = AdamW(model.parameters(), lr=2e-5)  # 定義優化器

    for epoch in range(EPOCHS):
        train_loss, train_accuracy = train_model(model, train_loader, optimizer, model.device)
        print(f'Epoch {epoch + 1}, Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}')

    model.save_pretrained('bert_chinese_model')  # 保存訓練好的模型

if __name__ == '__main__':
    main()


In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification

# 載入模型和分詞器
def load_model_and_tokenizer(model_path, tokenizer_name):
    model = BertForSequenceClassification.from_pretrained(model_path)
    tokenizer = BertTokenizer.from_pretrained(tokenizer_name)
    model.eval()  # 將模型設定為評估模式
    return model, tokenizer

# 预处理文本
def preprocess_text(text, tokenizer, max_len=128):
    encodings = tokenizer(text, truncation=True, padding='max_length', max_length=max_len, return_tensors='pt')
    return encodings

# 预测文本情感
def predict_sentiment(text, model, tokenizer):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    encodings = preprocess_text(text, tokenizer)
    with torch.no_grad():
        inputs = {key: val.to(device) for key, val in encodings.items()}
        outputs = model(**inputs)
        prediction = torch.argmax(outputs.logits, dim=-1)

    print(prediction)
    if prediction.item() == 2:
        return "正評"

    elif prediction.item() == 1:
        return "負評"

    else:
        return "中立"

# 主功能
def main():
    model_path = 'bert_chinese_model'
    tokenizer_name = 'bert-base-chinese'
    model, tokenizer = load_model_and_tokenizer(model_path, tokenizer_name)
    new_text = "我愛你"
    sentiment = predict_sentiment(new_text, model, tokenizer)
    print(f"預測的文本情感: {sentiment}")

if __name__ == '__main__':
    main()
